# Uncertainty-Aware Prediction of Asphalt Pavement Roughness Using Probabilistic Machine Learning

**Companion code repository**

This notebook accompanies the manuscript *"Uncertainty-Aware Prediction of Asphalt Pavement Roughness Using Probabilistic Machine Learning."* It documents the full modeling pipeline used to predict the International Roughness Index (IRI) of asphalt pavements from Long-Term Pavement Performance (LTPP) data integrated with NASA MERRA-2 climate variables and traffic loading metrics.

---

### What this notebook does

The pipeline trains and compares six machine learning models under one consistent framework:

| Type | Models |
|------|--------|
| Deterministic benchmarks | Random Forest (RF), Extreme Gradient Boosting (XGBoost), Artificial Neural Network (ANN), Support Vector Regression (SVR) |
| Probabilistic models | Bayesian Neural Network (BNN) with Monte Carlo dropout, Gaussian Process Regression (GPR) |

Every model is tuned with **Optuna** (TPE sampler) using **5-fold stratified cross-validation**, evaluated on a held-out test set, and (for the probabilistic models) assessed with uncertainty-calibration metrics (PICP, MPIW, CWC, NLL, ECE).

### Notebook structure

1. Setup and imports
2. Data preparation
3. Model training and tuning (RF, XGBoost, ANN, SVR, BNN, GPR)
4. Model comparison
5. Feature importance (SHAP)
6. Uncertainty and prediction intervals
7. Spatial and climate-region analysis

### Reproducibility note

This repository shares the **modeling code** for transparency and review. The LTPP-derived dataset and the trained model files are **not** included; they are available from the corresponding author oupon request. To run the notebook, place a `Model_Ready_Dataset_With_Climate_Label.xlsx` file (with the columns listed in the data-preparation section) in the path defined by `DATA_PATH`.

### Citation

If you use this code, please cite the associated paper.


## 1. Setup and Imports

This section installs and imports the required libraries, fixes the random seed for reproducibility, and defines small helper functions used throughout the notebook. All file paths are kept relative so the notebook can be run in any environment.

In [ ]:
# Install dependencies (uncomment if running in a fresh environment)
# !pip install optuna xgboost shap scikit-learn tensorflow openpyxl seaborn

In [ ]:
# -----------------------------------------------------------------------------
# Imports and global configuration
# -----------------------------------------------------------------------------
import os
import gc
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, WhiteKernel
from xgboost import XGBRegressor

import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, backend as K

import optuna
import shap
from scipy.stats import norm

# -----------------------------------------------------------------------------
# Configuration
# -----------------------------------------------------------------------------
RANDOM_STATE = 42
DATA_PATH    = "Model_Ready_Dataset_With_Climate_Label.xlsx"   # <-- place dataset here
RESULTS_DIR  = "results"
N_TRIALS     = 30

os.makedirs(RESULTS_DIR, exist_ok=True)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
tf.get_logger().setLevel("ERROR")
optuna.logging.set_verbosity(optuna.logging.WARNING)


def print_header(title, width=80):
    """Print a formatted section header."""
    print("\n" + "=" * width)
    print(title.upper().center(width))
    print("=" * width)


def print_subheader(title, width=80):
    """Print a formatted subheader."""
    print("\n" + "-" * width)
    print(title.center(width))
    print("-" * width)

print("Environment ready. TensorFlow", tf.__version__, "| SHAP", shap.__version__)

## 2. Data Preparation

The model-ready dataset combines pavement structural and construction variables, surface distress measures, traffic loading metrics, and MERRA-2 climate features, with the IRI as the prediction target and an LTPP climate-region label used for stratification.

| Group | Variables |
|-------|-----------|
| Traffic | `AADTT`, `KESAL` |
| Structure / construction | `AT`, `AV`, `AC`, `Gmm`, `Age`, `IRI0` |
| Surface distress | `RUT`, `CRA`, `CRT`, `CRL`, `CRW` |
| Climate (MERRA-2) | `PP`, `Tmax`, `Tmin`, `FI`, `WV`, `CC`, `EM` |
| Target | `IRI` |

Features are standardized with `StandardScaler`, and the data is split 80/20 into training and test sets with stratification on the climate region so that every region is represented in both subsets.

In [ ]:
# -----------------------------------------------------------------------------
# Load and prepare the dataset
# -----------------------------------------------------------------------------
print_header("data preparation")

df = pd.read_excel(DATA_PATH)
print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} columns")

COLUMNS = ["AADTT", "AT", "AV", "AC", "Gmm", "RUT", "CRA", "CRT", "CRL", "CRW",
           "PP", "Tmax", "Tmin", "FI", "WV", "CC", "EM", "Age", "IRI0",
           "IRI", "Climate_Region"]
df = df[COLUMNS]

print("\nClimate-region distribution:")
print(df["Climate_Region"].value_counts().sort_index())

X = df.drop(columns=["IRI", "Climate_Region"])
y = df["IRI"]
climate_regions = df["Climate_Region"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test, climate_train, climate_test = train_test_split(
    X_scaled, y, climate_regions,
    test_size=0.2, stratify=climate_regions, random_state=RANDOM_STATE
)

FEATURE_NAMES = list(X.columns)
print(f"\nTrain shape: {X_train.shape} | Test shape: {X_test.shape}")

## 3. Model Training and Hyperparameter Tuning

Each model is optimized with **Optuna** using the Tree-structured Parzen Estimator (TPE) sampler. For every model, the Optuna objective performs **5-fold stratified cross-validation** on the training set and returns the mean RMSE across folds, so the selected configuration reflects generalization rather than a single split. The best configuration is then retrained on the full training set and evaluated on the held-out test set.

The shared tuning helper below keeps the six model sections concise and consistent.

In [ ]:
# =============================================================================
# Model Definitions
# =============================================================================
print_header("model definitions")
from tensorflow.keras import layers, models
import tensorflow as tf

@tf.keras.utils.register_keras_serializable()
class MCDropout(layers.Dropout):
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)

def create_bnn_model(input_dim, hidden_units=64, dropout_rate=0.2, learning_rate=0.001):
    inputs = layers.Input(shape=(input_dim,))
    x = layers.Dense(hidden_units, activation='relu')(inputs)
    x = MCDropout(dropout_rate)(x)
    x = layers.Dense(hidden_units, activation='relu')(x)
    x = MCDropout(dropout_rate)(x)
    outputs = layers.Dense(1)(x)

    model = models.Model(inputs, outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss='mse',
                  metrics=['mae'])
    return model

def create_ann_model(input_dim, hidden_units=64, learning_rate=0.001):
    model = models.Sequential([
        layers.Dense(hidden_units, activation='relu', input_shape=(input_dim,)),
        layers.Dense(hidden_units, activation='relu'),
        layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss='mse',
                  metrics=['mae'])
    return model

def create_xgboost_model():
    return XGBRegressor(
        n_estimators=1000,
        learning_rate=0.01,
        max_depth=6,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

def create_rf_model():
    return RandomForestRegressor(
        n_estimators=500,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )


def create_gpr_model(length_scale=1.0, noise_level=1.0, alpha=1e-10, random_state=0):
    kernel = RBF(length_scale=length_scale) + WhiteKernel(noise_level=noise_level)
    return GaussianProcessRegressor(kernel=kernel, alpha=alpha, normalize_y=True, random_state=random_state)

print("Model architectures defined:")
print("- Bayesian Neural Network (BNN) with MC Dropout")
print("- Standard Artificial Neural Network (ANN)")
print("- XGBoost Regressor")
print("- Random Forest Regressor")
print("- Gaussian Process Regressor (GPR)")

In [ ]:
# =============================================================================
# Helper Functions
# =============================================================================
def predict_mc(model, X, T=50):
    """Monte Carlo prediction for BNN models"""
    return np.array([model.predict(X, batch_size=1024, verbose=0).squeeze() for _ in range(T)])

def vectorized_sensitivity_analysis(model, X_test, feature_names, target_feature, n_steps=20):
    """Optimized sensitivity analysis with vectorization"""
    feature_idx = feature_names.index(target_feature)
    baseline = np.median(X_test, axis=0)
    values = np.linspace(np.min(X_test[:, feature_idx]),
                         np.max(X_test[:, feature_idx]),
                         n_steps)

    test_matrix = np.tile(baseline, (n_steps, 1))
    test_matrix[:, feature_idx] = values

    if isinstance(model, (tf.keras.Model, tf.keras.Sequential)):
        predictions = model.predict(test_matrix, verbose=0, batch_size=n_steps).flatten()
    else:
        predictions = model.predict(test_matrix)

    return values, predictions

def stratified_sample_idx(labels, sample_size):
    """Generate stratified sample indices based on class labels"""
    unique_labels = np.unique(labels)
    sample_idx = []

    for label in unique_labels:
        label_idx = np.where(labels == label)[0]
        n_samples = max(1, int(sample_size * len(label_idx) / len(labels)))
        sample_idx.extend(np.random.choice(label_idx,
                                          size=min(n_samples, len(label_idx)),
                                          replace=False))
    return np.array(sample_idx)

def display_metrics(y_true, y_pred, title="Model Performance"):
    """Display performance metrics in formatted table"""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)

    metrics_df = pd.DataFrame({
        'Metric': ['RMSE', 'MAE', 'R²'],
        'Value': [rmse, mae, r2]
    })

    print(f"\n{title}:")
    print(metrics_df.to_string(index=False))
    print("\n" + "="*80)
    return rmse, mae, r2

In [ ]:
# -----------------------------------------------------------------------------
# Shared 5-fold cross-validation helper for scikit-learn-style models
# -----------------------------------------------------------------------------
def cv_rmse(model_fn, params):
    """Mean RMSE across 5 stratified folds for a model built by model_fn(params)."""
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rmses = []
    for tr_idx, val_idx in kfold.split(X_train, climate_train):
        model = model_fn(params)
        model.fit(X_train[tr_idx], y_train.iloc[tr_idx])
        pred = model.predict(X_train[val_idx])
        rmses.append(np.sqrt(mean_squared_error(y_train.iloc[val_idx], pred)))
    return float(np.mean(rmses))


def evaluate_and_store(model, name):
    """Fit on full training set, report train/test metrics, return a results dict."""
    model.fit(X_train, y_train)
    y_pred_train = model.predict(X_train)
    y_pred_test  = model.predict(X_test)
    display_metrics(y_train, y_pred_train, f"{name} (train)")
    display_metrics(y_test,  y_pred_test,  f"{name} (test)")
    return {
        "model_type": name,
        "models": [model],
        "y_true_train": y_train.values, "y_pred_train_mean": y_pred_train,
        "y_true_test": y_test.values,   "y_pred_test_mean": y_pred_test,
    }

### 3.1 Random Forest (RF)

Search space: number of estimators, maximum depth, minimum samples per split and leaf, and the maximum number of features per split.

In [ ]:
# =============================================================================
# Random Forest with Optuna Optimization
# =============================================================================
import optuna
from sklearn.model_selection import StratifiedKFold
import numpy as np
from sklearn.ensemble import RandomForestRegressor

print_header("random forest implementation with optuna")

def rf_objective(trial):
    n_estimators = trial.suggest_int("n_estimators", 200, 1000, step=100)
    max_depth = trial.suggest_int("max_depth", 6, 20)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 10)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 5)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2", None])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rmses = []
    for train_idx, val_idx in kfold.split(X_train, climate_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))
    return float(np.mean(rmses))

print_subheader("optuna tuning (this may take some time)")
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study_rf = optuna.create_study(direction="minimize", sampler=sampler, study_name="rf_opt")
study_rf.optimize(rf_objective, n_trials=30, n_jobs=1)
print("\nBest hyperparameters (Optuna):")
print(study_rf.best_params)
print(f"Best CV objective (mean RMSE): {study_rf.best_value:.6f}")

In [ ]:
print_subheader("cross-validation (best params)")
best_params = study_rf.best_params
cv_fold_stats = {'rmse': [], 'mae': [], 'r2': []}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, climate_train), start=1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_fold = RandomForestRegressor(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
    model_fold.fit(X_tr, y_tr)
    y_pred_val = model_fold.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    mae = mean_absolute_error(y_val, y_pred_val)
    r2 = r2_score(y_val, y_pred_val)

    cv_fold_stats['rmse'].append(rmse)
    cv_fold_stats['mae'].append(mae)
    cv_fold_stats['r2'].append(r2)

    print(f"\nFold {fold}/5")
    print(f"Fold {fold} - RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

print("\n" + "-"*80)
print("CV (5-fold) summary (mean ± std):")
print(f"RMSE: {np.mean(cv_fold_stats['rmse']):.4f} ± {np.std(cv_fold_stats['rmse']):.4f}")
print(f"MAE : {np.mean(cv_fold_stats['mae']):.4f} ± {np.std(cv_fold_stats['mae']):.4f}")
print(f"R²  : {np.mean(cv_fold_stats['r2']):.4f} ± {np.std(cv_fold_stats['r2']):.4f}")
print("-"*80)

print_subheader("training final random forest model with best params")
final_rf = RandomForestRegressor(**best_params, random_state=RANDOM_STATE, n_jobs=-1)
final_rf.fit(X_train, y_train)

y_pred_train = final_rf.predict(X_train)
y_pred_test  = final_rf.predict(X_test)

rf_results = {
    'model_type': 'RandomForest',
    'models': [final_rf],
    'best_params': best_params,
    'y_pred_train_mean': y_pred_train,
    'y_true_train': y_train.values,
    'y_pred_test_mean': y_pred_test,
    'y_true_test': y_test.values,
    'cv_fold_stats': cv_fold_stats,
    'cv_score_mean_rmse': float(np.mean(cv_fold_stats['rmse']))
}

rf_train_rmse, rf_train_mae, rf_train_r2 = display_metrics(y_train, y_pred_train, "Random Forest Overall Performance (TRAIN)")
rf_test_rmse, rf_test_mae, rf_test_r2 = display_metrics(y_test, y_pred_test, "Random Forest Overall Performance (TEST)")

print_subheader("regional performance - random forest (TEST)")
for region in sorted(climate_test.unique()):
    region_mask = (np.asarray(climate_test) == region)
    n_region = int(region_mask.sum())
    y_test_region = np.asarray(y_test)[region_mask]
    y_pred_region = np.asarray(y_pred_test)[region_mask]

    rmse = np.sqrt(mean_squared_error(y_test_region, y_pred_region))
    mae = mean_absolute_error(y_test_region, y_pred_region)
    r2 = r2_score(y_test_region, y_pred_region)

    print(f"\nRegion {region} (n={n_region}):")
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

print("\n" + "="*80)

### 3.2 Extreme Gradient Boosting (XGBoost)

Search space: number of estimators, learning rate (log scale), maximum depth, subsample and column-sample ratios, gamma, and minimum child weight.

In [ ]:
# =============================================================================
# XGBoost with Optuna Optimization
# =============================================================================
import optuna
import numpy as np
from sklearn.model_selection import StratifiedKFold
from xgboost import XGBRegressor

print_header("xgboost implementation with optuna")

def xgb_objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 1000, step=100),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.3, log=True),
        "max_depth": trial.suggest_int("max_depth", 3, 12),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
        "gamma": trial.suggest_float("gamma", 0, 5.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
    }

    model = XGBRegressor(**params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rmses = []
    for train_idx, val_idx in kfold.split(X_train, climate_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)
        y_pred = model.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))
    return float(np.mean(rmses))

print_subheader("optuna tuning (this may take some time)")
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study_xgb = optuna.create_study(direction="minimize", sampler=sampler, study_name="xgb_opt")
study_xgb.optimize(xgb_objective, n_trials=30, n_jobs=1)
print("\nBest hyperparameters (Optuna):")
print(study_xgb.best_params)
print(f"Best CV objective (mean RMSE): {study_xgb.best_value:.6f}")

In [ ]:
print_subheader("cross-validation (best params)")
best_params = study_xgb.best_params
cv_fold_stats = {'rmse': [], 'mae': [], 'r2': []}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, climate_train), start=1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_fold = XGBRegressor(**best_params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
    model_fold.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=0)

    y_pred_val = model_fold.predict(X_val)
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    mae = mean_absolute_error(y_val, y_pred_val)
    r2 = r2_score(y_val, y_pred_val)

    cv_fold_stats['rmse'].append(rmse)
    cv_fold_stats['mae'].append(mae)
    cv_fold_stats['r2'].append(r2)

    print(f"\nFold {fold}/5")
    print(f"Fold {fold} - RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

# CV summary
print("\n" + "-"*80)
print("CV (5-fold) summary (mean ± std):")
print(f"RMSE: {np.mean(cv_fold_stats['rmse']):.4f} ± {np.std(cv_fold_stats['rmse']):.4f}")
print(f"MAE : {np.mean(cv_fold_stats['mae']):.4f} ± {np.std(cv_fold_stats['mae']):.4f}")
print(f"R²  : {np.mean(cv_fold_stats['r2']):.4f} ± {np.std(cv_fold_stats['r2']):.4f}")
print("-"*80)

print_subheader("training final xgboost model with best params")
final_xgb = XGBRegressor(**best_params, random_state=RANDOM_STATE, n_jobs=-1, verbosity=0)
final_xgb.fit(X_train, y_train)

y_pred_train = final_xgb.predict(X_train)
y_pred_test  = final_xgb.predict(X_test)

xgb_results = {
    'model_type': 'XGBoost',
    'models': [final_xgb],
    'best_params': best_params,
    'y_pred_train_mean': y_pred_train,
    'y_true_train': y_train.values,
    'y_pred_test_mean': y_pred_test,
    'y_true_test': y_test.values,
    'cv_fold_stats': cv_fold_stats,
    'cv_score_mean_rmse': float(np.mean(cv_fold_stats['rmse']))
}

xgb_train_rmse, xgb_train_mae, xgb_train_r2 = display_metrics(y_train, y_pred_train, "XGBoost Overall Performance (TRAIN)")
xgb_test_rmse, xgb_test_mae, xgb_test_r2 = display_metrics(y_test, y_pred_test, "XGBoost Overall Performance (TEST)")

print_subheader("regional performance - xgboost (TEST)")
for region in sorted(climate_test.unique()):
    region_mask = (np.asarray(climate_test) == region)
    n_region = int(region_mask.sum())
    y_test_region = np.asarray(y_test)[region_mask]
    y_pred_region = np.asarray(y_pred_test)[region_mask]

    rmse = np.sqrt(mean_squared_error(y_test_region, y_pred_region))
    mae = mean_absolute_error(y_test_region, y_pred_region)
    r2 = r2_score(y_test_region, y_pred_region)

    print(f"\nRegion {region} (n={n_region}):")
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

print("\n" + "="*80)

### 3.3 Support Vector Regression (SVR)

Search space: kernel type, regularization parameter `C` (log scale), `epsilon` (log scale), and the kernel coefficient `gamma`.

In [ ]:
# =============================================================================
# SVR with Optuna Optimization
# =============================================================================
import optuna
from sklearn.svm import SVR
from sklearn.model_selection import StratifiedKFold
import numpy as np

print_header("svr implementation with optuna")

def svr_objective(trial):
    kernel = trial.suggest_categorical("kernel", ["rbf", "poly", "sigmoid"])
    C = trial.suggest_float("C", 1e-2, 1e3, log=True)
    epsilon = trial.suggest_float("epsilon", 1e-3, 1.0, log=True)
    gamma = trial.suggest_categorical("gamma", ["scale", "auto"])

    model = SVR(kernel=kernel, C=C, epsilon=epsilon, gamma=gamma)

    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rmses = []
    for train_idx, val_idx in kfold.split(X_train, climate_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_val)
        rmses.append(np.sqrt(mean_squared_error(y_val, y_pred)))
    return float(np.mean(rmses))

print_subheader("optuna tuning (this may take some time)")
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study_svr = optuna.create_study(direction="minimize", sampler=sampler, study_name="svr_opt")
study_svr.optimize(svr_objective, n_trials=30, n_jobs=1)

print("\nBest hyperparameters (Optuna):")
print(study_svr.best_params)
print(f"Best CV objective (mean RMSE): {study_svr.best_value:.6f}")

In [ ]:
print_subheader("cross-validation (best params)")

best_params = study_svr.best_params
model_best = SVR(**best_params)

kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_fold_stats = {'rmse': [], 'mae': [], 'r2': []}

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, climate_train), start=1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model_best.fit(X_tr, y_tr)
    y_pred_val = model_best.predict(X_val)

    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    mae = mean_absolute_error(y_val, y_pred_val)
    r2 = r2_score(y_val, y_pred_val)

    cv_fold_stats['rmse'].append(rmse)
    cv_fold_stats['mae'].append(mae)
    cv_fold_stats['r2'].append(r2)

    print(f"\nFold {fold}/5")
    print(f"Fold {fold} - RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

print("\n" + "-"*80)
print("CV (5-fold) summary (mean ± std):")
print(f"RMSE: {np.mean(cv_fold_stats['rmse']):.4f} ± {np.std(cv_fold_stats['rmse']):.4f}")
print(f"MAE : {np.mean(cv_fold_stats['mae']):.4f} ± {np.std(cv_fold_stats['mae']):.4f}")
print(f"R²  : {np.mean(cv_fold_stats['r2']):.4f} ± {np.std(cv_fold_stats['r2']):.4f}")
print("-"*80)

print_subheader("training final svr model with best params")
final_svr = SVR(**best_params)
final_svr.fit(X_train, y_train)

y_pred_train = final_svr.predict(X_train)
y_pred_test  = final_svr.predict(X_test)

svr_results = {
    'model_type': 'SVR',
    'models': [final_svr],
    'best_params': best_params,
    'y_pred_train_mean': y_pred_train,
    'y_true_train': y_train.values,
    'y_pred_test_mean': y_pred_test,
    'y_true_test': y_test.values,
    'cv_fold_stats': cv_fold_stats,
    'cv_score_mean_rmse': float(np.mean(cv_fold_stats['rmse']))
}

svr_train_rmse, svr_train_mae, svr_train_r2 = display_metrics(y_train, y_pred_train, "SVR Overall Performance (TRAIN)")
svr_test_rmse, svr_test_mae, svr_test_r2 = display_metrics(y_test, y_pred_test, "SVR Overall Performance (TEST)")

print_subheader("regional performance - svr (TEST)")
for region in sorted(climate_test.unique()):
    region_mask = (np.asarray(climate_test) == region)
    n_region = int(region_mask.sum())
    y_test_region = np.asarray(y_test)[region_mask]
    y_pred_region = np.asarray(y_pred_test)[region_mask]

    rmse = np.sqrt(mean_squared_error(y_test_region, y_pred_region))
    mae = mean_absolute_error(y_test_region, y_pred_region)
    r2 = r2_score(y_test_region, y_pred_region)

    print(f"\nRegion {region} (n={n_region}):")
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

print("\n" + "="*80)

### 3.4 Artificial Neural Network (ANN)

A feed-forward network with two hidden layers and dropout. Search space: hidden-layer width, dropout rate, learning rate (log scale), and batch size. Early stopping on the validation loss is used within each fold.

In [ ]:
# =============================================================================
# ANN with Optuna Optimization
# =============================================================================
import gc
from tensorflow.keras import layers, callbacks, backend as K
from sklearn.model_selection import StratifiedKFold

print_header("ann implementation with optuna")

def build_ann_model(input_dim, hidden_units, dropout_rate, learning_rate):
    model = tf.keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(hidden_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(hidden_units, activation='relu'),
        layers.Dropout(dropout_rate),
        layers.Dense(1)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate),
                  loss='mse',
                  metrics=['mae'])
    return model

def ann_objective(trial):
    hidden_units = trial.suggest_int("hidden_units", 32, 256, step=32)
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    epochs = 100
    patience = 10

    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    fold_scores = []

    for train_idx, val_idx in kfold.split(X_train, climate_train):
        K.clear_session(); gc.collect()

        X_tr_f, X_val_f = X_train[train_idx], X_train[val_idx]
        y_tr_f = y_train.iloc[train_idx].values.astype('float32')
        y_val_f = y_train.iloc[val_idx].values.astype('float32')

        model = build_ann_model(X_train.shape[1],
                                hidden_units=hidden_units,
                                dropout_rate=dropout_rate,
                                learning_rate=learning_rate)

        es = callbacks.EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True, verbose=0)

        model.fit(X_tr_f, y_tr_f,
                  validation_data=(X_val_f, y_val_f),
                  epochs=epochs,
                  batch_size=batch_size,
                  callbacks=[es],
                  verbose=0)

        y_val_pred = model.predict(X_val_f, batch_size=batch_size, verbose=0).flatten()
        rmse = np.sqrt(mean_squared_error(y_val_f, y_val_pred))
        fold_scores.append(rmse)

        K.clear_session(); gc.collect()

    return float(np.mean(fold_scores))

print_subheader("optuna tuning (this may take some time)")
sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study_ann = optuna.create_study(direction="minimize", sampler=sampler, study_name="ann_opt")
study_ann.optimize(ann_objective, n_trials=30, n_jobs=1)

print("\nBest hyperparameters (Optuna):")
print(study_ann.best_params)
print(f"Best CV objective (mean RMSE): {study_ann.best_value:.6f}")

In [ ]:
print_subheader("cross-validation (best params)")

import random
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

best = study_ann.best_params
cv_fold_stats = {'rmse': [], 'mae': [], 'r2': []}
kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for fold, (train_idx, val_idx) in enumerate(kfold.split(X_train, climate_train), start=1):
    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr = y_train.iloc[train_idx].values.astype('float32')
    y_val = y_train.iloc[val_idx].values.astype('float32')

    K.clear_session(); gc.collect()
    model_fold = build_ann_model(
        input_dim=X_train.shape[1],
        hidden_units=best.get('hidden_units', 128),
        dropout_rate=best.get('dropout_rate', 0.0),
        learning_rate=best.get('learning_rate', 1e-3)
    )

    model_fold.fit(
        X_tr, y_tr,
        validation_data=(X_val, y_val),
        epochs=100,
        batch_size=best.get('batch_size', 128),
        verbose=0
    )

    y_pred_val = model_fold.predict(X_val, batch_size=best.get('batch_size', 128), verbose=0).flatten()
    rmse = np.sqrt(mean_squared_error(y_val, y_pred_val))
    mae = mean_absolute_error(y_val, y_pred_val)
    r2 = r2_score(y_val, y_pred_val)

    cv_fold_stats['rmse'].append(rmse)
    cv_fold_stats['mae'].append(mae)
    cv_fold_stats['r2'].append(r2)

    print(f"\nFold {fold}/5")
    print(f"Fold {fold} - RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

# CV summary
print("\n" + "-"*80)
print("CV (5-fold) summary (mean ± std):")
print(f"RMSE: {np.mean(cv_fold_stats['rmse']):.4f} ± {np.std(cv_fold_stats['rmse']):.4f}")
print(f"MAE : {np.mean(cv_fold_stats['mae']):.4f} ± {np.std(cv_fold_stats['mae']):.4f}")
print(f"R²  : {np.mean(cv_fold_stats['r2']):.4f} ± {np.std(cv_fold_stats['r2']):.4f}")
print("-"*80)

print_subheader("training final ann model with best params (full epochs, no EarlyStopping)")
K.clear_session(); gc.collect()

random.seed(RANDOM_STATE); np.random.seed(RANDOM_STATE); tf.random.set_seed(RANDOM_STATE)

final_model = build_ann_model(
    input_dim=X_train.shape[1],
    hidden_units=best.get('hidden_units', 128),
    dropout_rate=best.get('dropout_rate', 0.0),
    learning_rate=best.get('learning_rate', 1e-3)
)

final_model.fit(
    X_train, y_train.values.astype('float32'),
    validation_split=0.05,
    epochs=200,
    batch_size=best.get('batch_size', 128),
    verbose=1
)

y_pred_train = final_model.predict(X_train, batch_size=best.get('batch_size', 128), verbose=0).flatten()
y_pred_test  = final_model.predict(X_test,  batch_size=best.get('batch_size', 128), verbose=0).flatten()

ann_results = {
    'model_type': 'ANN',
    'models': [final_model],
    'best_params': best,
    'y_pred_train_mean': y_pred_train,
    'y_true_train': y_train.values,
    'y_pred_test_mean': y_pred_test,
    'y_true_test': y_test.values,
    'cv_fold_stats': cv_fold_stats,
    'cv_score_mean_rmse': float(np.mean(cv_fold_stats['rmse']))
}

ann_train_rmse, ann_train_mae, ann_train_r2 = display_metrics(y_train, y_pred_train, "ANN Overall Performance (TRAIN)")
ann_test_rmse, ann_test_mae, ann_test_r2 = display_metrics(y_test, y_pred_test, "ANN Overall Performance (TEST)")

print_subheader("regional performance - ann (TEST)")
for region in sorted(climate_test.unique()):
    region_mask = (np.asarray(climate_test) == region)
    n_region = int(region_mask.sum())
    y_test_region = np.asarray(y_test)[region_mask]
    y_pred_region = np.asarray(y_pred_test)[region_mask]

    rmse = np.sqrt(mean_squared_error(y_test_region, y_pred_region))
    mae = mean_absolute_error(y_test_region, y_pred_region)
    r2 = r2_score(y_test_region, y_pred_region)

    print(f"\nRegion {region} (n={n_region}):")
    print(f"RMSE: {rmse:.4f} | MAE: {mae:.4f} | R²: {r2:.4f}")

print("\n" + "="*80)

### 3.5 Bayesian Neural Network (BNN) with Monte Carlo Dropout

The BNN uses dropout layers that remain **active at inference**, so repeated forward passes (Monte Carlo sampling) yield a predictive distribution rather than a single value. The predictive mean is the point estimate, and the spread across passes quantifies uncertainty. To stabilize the estimates, an **ensemble of five independently trained networks** is used and their predictions averaged. The target is standardized during training and back-transformed for evaluation.

Search space: hidden-layer width, dropout rate, and learning rate (log scale).

In [ ]:
# -----------------------------------------------------------------------------
print_header("BNN - with Optuna optimization")
# -----------------------------------------------------------------------------
# Reuses the imports, seed, and helper functions (print_header, print_subheader,
# display_metrics) defined in Section 1. Only the BNN model constructor, the
# Monte Carlo prediction helper, and the target scaling are defined here.

y_mean = y_train.mean()
y_std = y_train.std()
y_train_scaled = (y_train - y_mean) / y_std


def create_bnn_model(input_dim, hidden_units, dropout_rate, learning_rate):
    """Feed-forward network with always-on (Monte Carlo) dropout."""
    inputs = tf.keras.Input(shape=(input_dim,))
    x = tf.keras.layers.Dense(hidden_units, activation="relu")(inputs)
    x = tf.keras.layers.Dropout(dropout_rate)(x, training=True)
    x = tf.keras.layers.Dense(hidden_units, activation="relu")(x)
    x = tf.keras.layers.Dropout(dropout_rate)(x, training=True)
    outputs = tf.keras.layers.Dense(1)(x)
    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate), loss="mse")
    return model


def predict_mc(model, X, T=100):
    """Monte Carlo prediction: T stochastic forward passes with dropout active."""
    return np.stack([model(X, training=True).numpy().squeeze() for _ in range(T)])

In [ ]:
# ----------------------------
# Cell 2: Optuna tuning
# ----------------------------
print_subheader("Optuna tuning (5-fold CV with EarlyStopping)")

bnn_cv_fold_metrics = []

def bnn_objective(trial):
    hidden_units = trial.suggest_int('hidden_units', 32, 128)
    dropout_rate = trial.suggest_float('dropout_rate', 0.1, 0.5)
    learning_rate = trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True)

    kfold = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rmses, maes, r2s = [], [], []

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train, climate_train), 1):
        tf.keras.backend.clear_session(); gc.collect()
        model = create_bnn_model(
            input_dim=X_train.shape[1],
            hidden_units=hidden_units,
            dropout_rate=dropout_rate,
            learning_rate=learning_rate
        )

        early_stop = tf.keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5, restore_best_weights=True, verbose=0)

        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr = y_train_scaled.iloc[train_idx].values.astype("float32")
        y_val = y_train_scaled.iloc[val_idx].values.astype("float32")

        model.fit(X_tr, y_tr, epochs=100, batch_size=128, verbose=0,
                  validation_data=(X_val, y_val), callbacks=[early_stop])

        preds_mc = predict_mc(model, X_val, T=100).mean(axis=0)
        preds_mc = preds_mc * y_std + y_mean
        y_val_orig = y_train.iloc[val_idx].values

        rmse = np.sqrt(mean_squared_error(y_val_orig, preds_mc))
        mae = mean_absolute_error(y_val_orig, preds_mc)
        r2  = r2_score(y_val_orig, preds_mc)

        rmses.append(rmse)
        maes.append(mae)
        r2s.append(r2)

    if trial.number == 0:
        for i in range(5):
            bnn_cv_fold_metrics.append({
                "fold": i+1,
                "rmse": rmses[i],
                "mae": maes[i],
                "r2": r2s[i]
            })

    return float(np.mean(rmses))

sampler = optuna.samplers.TPESampler(seed=RANDOM_STATE)
study_bnn = optuna.create_study(direction="minimize", sampler=sampler)
study_bnn.optimize(bnn_objective, n_trials=20, n_jobs=1)

best = study_bnn.best_params
print("\nBest Hyperparameters:", best)

print_subheader("Cross-validation (best params)")
for item in bnn_cv_fold_metrics:
    print(f"Fold {item['fold']} - RMSE: {item['rmse']:.4f} | MAE: {item['mae']:.4f} | R²: {item['r2']:.4f}")

rmse_values = [m["rmse"] for m in bnn_cv_fold_metrics]
mae_values = [m["mae"] for m in bnn_cv_fold_metrics]
r2_values  = [m["r2"]  for m in bnn_cv_fold_metrics]

print("\n" + "-"*80)
print("CV (5-fold) summary (mean ± std):")
print(f"RMSE: {np.mean(rmse_values):.4f} ± {np.std(rmse_values):.4f}")
print(f"MAE : {np.mean(mae_values):.4f} ± {np.std(mae_values):.4f}")
print(f"R²  : {np.mean(r2_values):.4f} ± {np.std(r2_values):.4f}")
print("-"*80)


In [ ]:
# ----------------------------
# Cell 3: Final ensemble training & evaluation
# ----------------------------
print_subheader("Final Ensemble Training")

n_models = 5
ensemble = []
fold_metrics = []

for i in range(n_models):
    print(f"Training model {i+1}/{n_models}...")
    tf.keras.backend.clear_session(); gc.collect()
    model = create_bnn_model(
        input_dim=X_train.shape[1],
        hidden_units=best['hidden_units'],
        dropout_rate=best['dropout_rate'],
        learning_rate=best['learning_rate']
    )

    model.fit(X_train, y_train_scaled.values.astype("float32"),
              epochs=100, batch_size=128, verbose=0)
    ensemble.append(model)

    pred_train = predict_mc(model, X_train, T=100).mean(axis=0) * y_std + y_mean
    pred_test  = predict_mc(model, X_test,  T=100).mean(axis=0) * y_std + y_mean

    y_train_true = y_train.values
    y_test_true  = y_test.values

    rmse_train = np.sqrt(mean_squared_error(y_train_true, pred_train))
    mae_train = mean_absolute_error(y_train_true, pred_train)
    r2_train  = r2_score(y_train_true, pred_train)

    rmse_test = np.sqrt(mean_squared_error(y_test_true, pred_test))
    mae_test = mean_absolute_error(y_test_true, pred_test)
    r2_test  = r2_score(y_test_true, pred_test)

    fold_metrics.append({
        "fold": i+1,
        "train_rmse": rmse_train,
        "train_mae": mae_train,
        "train_r2": r2_train,
        "test_rmse": rmse_test,
        "test_mae": mae_test,
        "test_r2": r2_test
    })

print_subheader("Ensemble Fold-wise Results")
for m in fold_metrics:
    print(f"Model {m['fold']}: "
          f"Train RMSE={m['train_rmse']:.4f}, MAE={m['train_mae']:.4f}, R²={m['train_r2']:.4f} | "
          f"Test RMSE={m['test_rmse']:.4f}, MAE={m['test_mae']:.4f}, R²={m['test_r2']:.4f}")

T = 100
print_subheader("MC Prediction")
all_preds_test = np.array([predict_mc(m, X_test, T=T) for m in ensemble])
all_preds_train = np.array([predict_mc(m, X_train, T=T) for m in ensemble])

y_pred_test_mean = all_preds_test.mean(axis=(0,1)) * y_std + y_mean
y_pred_test_std  = all_preds_test.std(axis=(0,1)) * y_std

y_pred_train_mean = all_preds_train.mean(axis=(0,1)) * y_std + y_mean
y_pred_train_std  = all_preds_train.std(axis=(0,1)) * y_std

bnn_results = {
    'model_type': 'MC-BNN',
    'models': ensemble,
    'best_params': best,
    'fold_metrics': fold_metrics,
    'y_pred_test_mean': y_pred_test_mean,
    'y_pred_test_std': y_pred_test_std,
    'y_pred_train_mean': y_pred_train_mean,
    'y_true_train': y_train.values,
    'y_true_test': y_test.values,
    'predictions_test_mc': all_preds_test
}

bnn_train_rmse, bnn_train_mae, bnn_train_r2 = display_metrics(y_train, y_pred_train_mean, "TRAIN")
bnn_test_rmse,  bnn_test_mae,  bnn_test_r2  = display_metrics(y_test,  y_pred_test_mean,  "TEST")

print_subheader("Regional Performance (Test)")
regional_metrics = []
for region in sorted(np.unique(np.asarray(climate_test))):
    mask = np.asarray(climate_test) == region
    y_r = np.asarray(y_test)[mask]
    y_p = np.asarray(y_pred_test_mean)[mask]
    rmse = np.sqrt(mean_squared_error(y_r, y_p))
    mae = mean_absolute_error(y_r, y_p)
    r2 = r2_score(y_r, y_p)
    print(f"Region {region}: RMSE={rmse:.4f}, MAE={mae:.4f}, R²={r2:.4f}")
    regional_metrics.append({
        "region": region, "rmse": rmse, "mae": mae, "r2": r2, "samples": int(mask.sum())
    })

bnn_results["regional_metrics"] = regional_metrics

In [ ]:
# ----------------------------
# Cell 4: Full Uncertainty + Interval Metrics (90% and 95%)
# ----------------------------
from scipy.stats import norm
import matplotlib.pyplot as plt

print_subheader("Full Uncertainty Metrics (90% & 95%)")

mc_preds = bnn_results['predictions_test_mc']
y_true = bnn_results['y_true_test']
y_mean = bnn_results['y_pred_mean']
y_std = bnn_results['y_pred_std']
n_samples = len(y_true)

total_uncertainty = mc_preds.var(axis=(0, 1))
model_means = mc_preds.mean(axis=1)
epistemic_uncertainty = model_means.var(axis=0)
model_vars = mc_preds.var(axis=1)
aleatoric_uncertainty = model_vars.mean(axis=0)

bnn_results['total_uncertainty'] = total_uncertainty
bnn_results['epistemic_uncertainty'] = epistemic_uncertainty
bnn_results['aleatoric_uncertainty'] = aleatoric_uncertainty

print(f"Mean Total Uncertainty     : {np.mean(total_uncertainty):.6f}")
print(f"Mean Epistemic Uncertainty : {np.mean(epistemic_uncertainty):.6f}")
print(f"Mean Aleatoric Uncertainty : {np.mean(aleatoric_uncertainty):.6f}")

def compute_interval_metrics(z, label):
    lower = y_mean - z * y_std
    upper = y_mean + z * y_std
    width = upper - lower
    within = np.logical_and(y_true >= lower, y_true <= upper)

    picp = np.mean(within)
    mpiw = np.mean(width)
    gamma = norm.cdf(z) * 2
    eta = 50
    cwc = mpiw if picp >= gamma else mpiw * (1 + eta * (gamma - picp))

    bnn_results[f'ci_lower_{label}'] = lower
    bnn_results[f'ci_upper_{label}'] = upper
    bnn_results[f'picp_{label}'] = picp
    bnn_results[f'mpiw_{label}'] = mpiw
    bnn_results[f'cwc_{label}'] = cwc

    print(f"\n== {int(gamma*100)}% Confidence Interval ==")
    print(f"PICP_{label.upper()}: {picp:.4f}")
    print(f"MPIW_{label.upper()}: {mpiw:.4f}")
    print(f"CWC_{label.upper()} : {cwc:.4f}")

# 90% (z=1.645) and 95% (z=1.96)
compute_interval_metrics(1.645, "90")
compute_interval_metrics(1.96, "95")

nll = np.mean(0.5 * np.log(2 * np.pi * y_std**2) + ((y_true - y_mean)**2) / (2 * y_std**2))
bnn_results['nll'] = nll
print(f"\nNegative Log-Likelihood (NLL): {nll:.4f}")

def compute_ece(y_true, y_mean, y_std, n_bins=10):
    probs = norm.cdf((y_true - y_mean) / y_std)
    confidences = np.linspace(0.0, 1.0, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower = confidences[i]
        bin_upper = confidences[i+1]
        in_bin = (probs >= bin_lower) & (probs < bin_upper)
        if np.any(in_bin):
            acc = np.mean((y_true[in_bin] >= (y_mean[in_bin] - y_std[in_bin])) &
                          (y_true[in_bin] <= (y_mean[in_bin] + y_std[in_bin])))
            conf = np.mean(probs[in_bin])
            ece += np.abs(acc - conf) * len(probs[in_bin]) / len(probs)
    return ece

ece = compute_ece(y_true, y_mean, y_std)
bnn_results['ece'] = ece
print(f"Expected Calibration Error (ECE): {ece:.4f}")

### 3.6 Gaussian Process Regression (GPR)

GPR is a non-parametric kernel method that returns both a predictive mean and a predictive standard deviation directly. The kernel is a sum of a radial-basis-function (RBF) term and a `WhiteKernel` noise term, which captures the observational noise typical of LTPP measurements. An ensemble of five models with different seeds is averaged.

Search space: RBF length scale (log scale), white-noise level (log scale), and the jitter term `alpha` (log scale).

In [ ]:
print_header("GPR - with Optuna optimization")

def gpr_objective(trial):
    length_scale = trial.suggest_float('length_scale', 0.1, 10.0, log=True)
    noise_level  = trial.suggest_float('noise_level', 1e-3, 1.0, log=True)
    alpha        = trial.suggest_float('alpha', 1e-10, 1e-2, log=True)

    kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    rmses = []
    for train_idx, val_idx in kf.split(X_train, climate_train):
        gc.collect()
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

        model = create_gpr_model(
            length_scale=length_scale,
            noise_level=noise_level,
            alpha=alpha,
            random_state=RANDOM_STATE
        )
        model.fit(X_tr, y_tr)
        preds, _ = model.predict(X_val, return_std=True)
        rmse = np.sqrt(mean_squared_error(y_val, preds))
        rmses.append(rmse)

    return float(np.mean(rmses))

study_gpr = optuna.create_study(direction='minimize')
study_gpr.optimize(gpr_objective, n_trials=20)

best_params_gpr = study_gpr.best_params
print("\nBest hyperparameters (GPR):")
print(best_params_gpr)


In [ ]:
print_header("GPR - Train Ensemble with Best Params")

n_models = 5
gpr_ensemble = []

for i in range(n_models):
    print(f"Training GPR model {i+1}/{n_models}")
    model = create_gpr_model(
        length_scale=best_params_gpr['length_scale'],
        noise_level=best_params_gpr['noise_level'],
        alpha=best_params_gpr['alpha'],
        random_state=RANDOM_STATE + i
    )
    model.fit(X_train, y_train)
    gpr_ensemble.append(model)

print("✅ All GPR ensemble models trained.")

In [ ]:
print_header("GPR Final Evaluation")

preds_train_list, preds_test_list = [], []
for model in gpr_ensemble:
    p_train, _ = model.predict(X_train, return_std=True)
    p_test,  _ = model.predict(X_test,  return_std=True)
    preds_train_list.append(p_train)
    preds_test_list.append(p_test)

y_pred_train_mean = np.mean(preds_train_list, axis=0)
y_pred_test_mean  = np.mean(preds_test_list, axis=0)

y_train_np = np.array(y_train)
y_test_np = np.array(y_test)
climate_train_np = np.array(climate_train)
climate_test_np = np.array(climate_test)

print_subheader("Overall Performance")

metrics_overall = {
    "Split": ["Train", "Test"],
    "RMSE": [
        np.sqrt(mean_squared_error(y_train_np, y_pred_train_mean)),
        np.sqrt(mean_squared_error(y_test_np,  y_pred_test_mean))
    ],
    "MAE": [
        mean_absolute_error(y_train_np, y_pred_train_mean),
        mean_absolute_error(y_test_np,  y_pred_test_mean)
    ],
    "R²": [
        r2_score(y_train_np, y_pred_train_mean),
        r2_score(y_test_np,  y_pred_test_mean)
    ],
}
df_overall = pd.DataFrame(metrics_overall)
display(df_overall.style.format({'RMSE': '{:.4f}', 'MAE': '{:.4f}', 'R²': '{:.4f}'}))

def get_regional_metrics(y_true, y_pred, climate, split_name):
    results = []
    for region in sorted(np.unique(climate)):
        mask = (climate == region)
        results.append({
            "Region": region,
            "Samples": int(mask.sum()),
            "RMSE": np.sqrt(mean_squared_error(y_true[mask], y_pred[mask])),
            "MAE": mean_absolute_error(y_true[mask], y_pred[mask]),
            "R²": r2_score(y_true[mask], y_pred[mask]),
            "Split": split_name
        })
    return results

regional_train = get_regional_metrics(y_train_np, y_pred_train_mean, climate_train_np, "Train")
regional_test  = get_regional_metrics(y_test_np,  y_pred_test_mean,  climate_test_np,  "Test")
df_regional = pd.DataFrame(regional_train + regional_test)

print_subheader("Regional Performance by Climate Region")
display(df_regional.pivot(index="Region", columns="Split", values=["RMSE", "MAE", "R²"]).style.format("{:.4f}"))

print("\n✅ GPR evaluation complete.")

In [ ]:
print_header("GPR - True 5-Fold Cross-Validation Evaluation")

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
true_cv_results = []

for i, (train_idx, val_idx) in enumerate(kf.split(X_train, climate_train), 1):
    print(f"Fold {i} training...")

    X_tr, X_val = X_train[train_idx], X_train[val_idx]
    y_tr, y_val = y_train.iloc[train_idx], y_train.iloc[val_idx]

    model = create_gpr_model(
        length_scale=best_params_gpr['length_scale'],
        noise_level=best_params_gpr['noise_level'],
        alpha=best_params_gpr['alpha'],
        random_state=RANDOM_STATE + i
    )
    model.fit(X_tr, y_tr)

    preds, _ = model.predict(X_val, return_std=True)

    rmse = np.sqrt(mean_squared_error(y_val, preds))
    mae = mean_absolute_error(y_val, preds)
    r2 = r2_score(y_val, preds)

    true_cv_results.append({
        "Fold": i,
        "RMSE": rmse,
        "MAE": mae,
        "R²": r2
    })

df_true_cv = pd.DataFrame(true_cv_results)
display(df_true_cv.style.format({"RMSE": "{:.4f}", "MAE": "{:.4f}", "R²": "{:.4f}"}))

print("\nSummary (mean ± std):")
print(f"RMSE: {df_true_cv['RMSE'].mean():.4f} ± {df_true_cv['RMSE'].std():.4f}")
print(f"MAE : {df_true_cv['MAE'].mean():.4f} ± {df_true_cv['MAE'].std():.4f}")
print(f"R²  : {df_true_cv['R²'].mean():.4f} ± {df_true_cv['R²'].std():.4f}")

In [ ]:
print_header("GPR - Uncertainty Metrics (90% and 95% Prediction Intervals)")

from scipy.stats import norm

means = []
stds = []

for model in gpr_ensemble:
    mean_i, std_i = model.predict(X_test, return_std=True)
    means.append(mean_i)
    stds.append(std_i)

means = np.array(means)
stds = np.array(stds)

y_pred_test_mean = np.mean(means, axis=0)
y_pred_test_std  = np.std(means, axis=0)

epistemic_uncertainty  = np.std(means, axis=0)
aleatoric_uncertainty  = np.mean(stds**2, axis=0)**0.5
total_uncertainty      = np.sqrt(epistemic_uncertainty**2 + aleatoric_uncertainty**2)

def compute_uncertainty_metrics(y_true, y_pred_mean, y_pred_std, confidence_level=0.95):
    z = norm.ppf(0.5 + confidence_level / 2)
    lower = y_pred_mean - z * y_pred_std
    upper = y_pred_mean + z * y_pred_std

    within_interval = (y_true >= lower) & (y_true <= upper)
    picp = np.mean(within_interval)
    mpiw = np.mean(upper - lower)

    penalty = 0 if picp >= confidence_level else 100 * (confidence_level - picp)
    cwc = mpiw * (1 + penalty)

    nll = -np.mean(norm.logpdf(y_true, loc=y_pred_mean, scale=y_pred_std + 1e-6))

    num_bins = 10
    probs = norm.cdf(y_true, loc=y_pred_mean, scale=y_pred_std)
    bins = np.linspace(0, 1, num_bins + 1)
    bin_ids = np.digitize(probs, bins) - 1

    ece = 0
    for i in range(num_bins):
        in_bin = bin_ids == i
        if np.any(in_bin):
            avg_conf = probs[in_bin].mean()
            acc = np.mean((probs[in_bin] > 0.5) == (y_true[in_bin] > y_pred_mean[in_bin]))
            ece += np.abs(acc - avg_conf) * len(probs[in_bin]) / len(probs)

    return {
        "PICP": picp,
        "MPIW": mpiw,
        "CWC": cwc,
        "NLL": nll,
        "ECE": ece
    }

uncert_90 = compute_uncertainty_metrics(y_test, y_pred_test_mean, total_uncertainty, confidence_level=0.90)
uncert_95 = compute_uncertainty_metrics(y_test, y_pred_test_mean, total_uncertainty, confidence_level=0.95)

df_uncertainty = pd.DataFrame([uncert_90, uncert_95], index=["90% PI", "95% PI"])

print_subheader("📉 Uncertainty Metrics on Test Set")
display(df_uncertainty.style.format("{:.4f}"))

print_subheader("🔍 Mean Uncertainty Components (Test Set)")
print(f"Mean Total Uncertainty     : {np.mean(total_uncertainty):.6f}")
print(f"Mean Epistemic Uncertainty : {np.mean(epistemic_uncertainty):.6f}")
print(f"Mean Aleatoric Uncertainty : {np.mean(aleatoric_uncertainty):.6f}")

print("\n✅ GPR Uncertainty metrics calculated and displayed.")

gpr_results = {
    'model_type': 'GPR',
    'models': gpr_ensemble,
    'best_params': best_params_gpr,
    'y_true_train': np.array(y_train),
    'y_pred_train_mean': y_pred_train_mean,
    'y_true_test': np.array(y_test),
    'y_pred_test_mean': y_pred_test_mean,
    'y_pred_test_std': total_uncertainty,
}
print("\n✅ gpr_results assembled with standard keys.")

## 4. Model Comparison

The six models are compared on the held-out test set using RMSE, MAE, and R squared. Tree-based ensembles are expected to lead on point accuracy, while the probabilistic models add calibrated uncertainty.

In [ ]:
print_header("model comparison")

all_results = {
    "RF": rf_results, "XGBoost": xgb_results, "SVR": svr_results,
    "ANN": ann_results, "BNN": bnn_results, "GPR": gpr_results,
}

rows = []
for name, res in all_results.items():
    yt, yp = res["y_true_test"], res["y_pred_test_mean"]
    rows.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(yt, yp)),
        "MAE":  mean_absolute_error(yt, yp),
        "R2":   r2_score(yt, yp),
    })
comparison_df = pd.DataFrame(rows).sort_values("R2", ascending=False).reset_index(drop=True)
print(comparison_df.to_string(index=False))

## 5. Uncertainty and Prediction Intervals

For the probabilistic models, prediction intervals are formed as the predictive mean plus or minus *z* times the predictive standard deviation. Calibration is assessed with five metrics:

- **PICP** (Prediction Interval Coverage Probability) - fraction of true values that fall inside the interval.
- **MPIW** (Mean Prediction Interval Width) - average interval width.
- **CWC** (Coverage Width-based Criterion) - penalizes intervals that fail to reach the target coverage.
- **NLL** (Negative Log-Likelihood) - agreement between the predicted distribution and the observations.
- **ECE** (Expected Calibration Error) - deviation between nominal and empirical coverage.

In [ ]:
def uncertainty_metrics(y_true, mean, std, confidence=0.95, n_bins=10):
    """Return PICP, MPIW, CWC, NLL, ECE for a Gaussian predictive distribution."""
    z = norm.ppf(0.5 + confidence / 2)
    lower, upper = mean - z * std, mean + z * std
    within = (y_true >= lower) & (y_true <= upper)

    picp = np.mean(within)
    mpiw = np.mean(upper - lower)
    penalty = 0.0 if picp >= confidence else 100 * (confidence - picp)
    cwc = mpiw * (1 + penalty)
    nll = -np.mean(norm.logpdf(y_true, loc=mean, scale=std + 1e-6))

    probs = norm.cdf(y_true, loc=mean, scale=std + 1e-6)
    bins = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for lo, hi in zip(bins[:-1], bins[1:]):
        in_bin = (probs >= lo) & (probs < hi)
        if in_bin.any():
            ece += abs(np.mean(in_bin) - (hi - lo))
    return {"PICP": picp, "MPIW": mpiw, "CWC": cwc, "NLL": nll, "ECE": ece}


for name, res in [("BNN", bnn_results), ("GPR", gpr_results)]:
    m = uncertainty_metrics(res["y_true_test"], res["y_pred_test_mean"], res["y_pred_test_std"])
    print(f"{name}: " + " | ".join(f"{k}={v:.4f}" for k, v in m.items()))

## 6. Spatial and Climate-Region Analysis

This section examines how performance and pavement deterioration vary geographically. Test-set performance is broken down by the four LTPP climate regions, and (where state identifiers are available in the source data) state-level mean IRI and deterioration rates can be mapped to reveal regional patterns. The geographic heterogeneity motivates the climate-region-stratified evaluation and provides context for interpreting the regional uncertainty-calibration results.

In [ ]:
print_header("climate-region performance")

REGION_NAMES = {0: "Dry-Freeze (DF)", 1: "Dry-NoFreeze (DNF)",
                2: "Wet-Freeze (WF)", 3: "Wet-NoFreeze (WNF)"}

region_rows = []
ct = np.asarray(climate_test)
for name, res in all_results.items():
    yt, yp = np.asarray(res["y_true_test"]), np.asarray(res["y_pred_test_mean"])
    for r in sorted(np.unique(ct)):
        mask = ct == r
        if mask.sum() > 1:
            region_rows.append({
                "Model": name, "Region": REGION_NAMES.get(r, r),
                "RMSE": np.sqrt(mean_squared_error(yt[mask], yp[mask])),
                "R2":   r2_score(yt[mask], yp[mask]),
            })
region_df = pd.DataFrame(region_rows)
print(region_df.pivot(index="Region", columns="Model", values="R2").round(4).to_string())

---

*End of notebook. The modeling pipeline above reproduces the methodology described in the manuscript. Dataset and trained-model files are available from the corresponding author upon request.*